In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/customer-churn-dataset/customer_churn_dataset-testing-master.csv
/kaggle/input/customer-churn-dataset/customer_churn_dataset-training-master.csv


In [2]:
df = pd.read_csv('/kaggle/input/customer-churn-dataset/customer_churn_dataset-training-master.csv')
print(df.shape)
df.head()

(440833, 12)


,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,2.0,30.0,Female,39.0,14.0,5.0,18.0,Standard,Annual,932.0,17.0,1.0
1,3.0,65.0,Female,49.0,1.0,10.0,8.0,Basic,Monthly,557.0,6.0,1.0
2,4.0,55.0,Female,14.0,4.0,6.0,18.0,Basic,Quarterly,185.0,3.0,1.0
3,5.0,58.0,Male,38.0,21.0,7.0,7.0,Standard,Monthly,396.0,29.0,1.0
4,6.0,23.0,Male,32.0,20.0,5.0,8.0,Basic,Monthly,617.0,20.0,1.0


In [3]:
df.info()
df.describe()
df['Churn'].value_counts(normalize=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440833 entries, 0 to 440832
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   CustomerID         440832 non-null  float64
 1   Age                440832 non-null  float64
 2   Gender             440832 non-null  object 
 3   Tenure             440832 non-null  float64
 4   Usage Frequency    440832 non-null  float64
 5   Support Calls      440832 non-null  float64
 6   Payment Delay      440832 non-null  float64
 7   Subscription Type  440832 non-null  object 
 8   Contract Length    440832 non-null  object 
 9   Total Spend        440832 non-null  float64
 10  Last Interaction   440832 non-null  float64
 11  Churn              440832 non-null  float64
dtypes: float64(9), object(3)
memory usage: 40.4+ MB


Churn
1.0    0.567107
0.0    0.432893
Name: proportion, dtype: float64

In [4]:
df = df.drop(columns=['CustomerID'], errors='ignore')

cat_cols = df.select_dtypes(include='object').columns
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

df['Churn'] = df['Churn'].fillna(0).astype(int)

df = df.fillna(df.median(numeric_only=True))

In [5]:
leak_cols = ['Age','Support Calls','Payment Delay','Total Spend','Contract Length_Monthly']

X = df.drop(columns=['Churn'])
X = X.drop(columns=[c for c in leak_cols if c in X.columns])

y = df['Churn']

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

pipe_lr = Pipeline([('sc', StandardScaler()),('lr', LogisticRegression(class_weight='balanced',max_iter=1000,random_state=42))])

pipe_lr.fit(X_train, y_train)
y_prob_lr = pipe_lr.predict_proba(X_test)[:,1]
y_pred_lr = pipe_lr.predict(X_test)

print("ROC-AUC:", round(roc_auc_score(y_test, y_prob_lr),4))
print(classification_report(y_test, y_pred_lr))
print(confusion_matrix(y_test, y_pred_lr))

ROC-AUC: 0.6724
              precision    recall  f1-score   support

           0       0.56      0.63      0.59     38167
           1       0.69      0.61      0.65     50000

    accuracy                           0.62     88167
   macro avg       0.62      0.62      0.62     88167
weighted avg       0.63      0.62      0.62     88167

[[24157 14010]
 [19345 30655]]


In [8]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_prob_rf = rf.predict_proba(X_test)[:,1]
y_pred_rf = rf.predict(X_test)

print("ROC-AUC:", round(roc_auc_score(y_test, y_prob_rf),4))
print(classification_report(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))

ROC-AUC: 0.6871
              precision    recall  f1-score   support

           0       0.55      0.59      0.57     38167
           1       0.67      0.64      0.65     50000

    accuracy                           0.62     88167
   macro avg       0.61      0.61      0.61     88167
weighted avg       0.62      0.62      0.62     88167

[[22381 15786]
 [18033 31967]]


In [9]:
probs = y_prob_rf
risk = pd.cut(probs, bins=[0,0.33,0.66,1], labels=['Low','Medium','High'])
results = pd.DataFrame({'Probability': probs, 'Risk_Level': risk})
results.head()

,Probability,Risk_Level
0,1.000000,High
1,0.138897,Low
2,0.092028,Low
3,0.536650,Medium
4,0.345803,Medium
